<a href="https://colab.research.google.com/github/porquenogr/precision-livestock-ai/blob/main/06_annotation_conversion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "trainingdatapro/farm-animals-pigs-detection-dataset"
)

print("[INFO] Dataset downloaded to:", dataset_path)

100%|██████████| 5.15M/5.15M [00:00<00:00, 200MB/s]

Extracting files...
[INFO] Dataset downloaded to: /root/.cache/kagglehub/datasets/trainingdatapro/farm-animals-pigs-detection-dataset/versions/1


In [3]:
from pathlib import Path

source_path = Path(dataset_path)
xml_file = source_path / "annotations.xml"

print("[INFO] XML exists:", xml_file.exists())
print("[INFO] XML path:", xml_file)

[INFO] XML exists: True
[INFO] XML path: /root/.cache/kagglehub/datasets/trainingdatapro/farm-animals-pigs-detection-dataset/versions/1/annotations.xml


In [4]:
from xml.etree import ElementTree as ET

tree = ET.parse(xml_file)
root = tree.getroot()

first_image = root.findall("image")[0]

print("[INFO] Image:", first_image.get("name"))
print("[INFO] Size:", first_image.get("width"), "x", first_image.get("height"))

box = first_image.find("box")

print("[INFO] Label:", box.get("label"))
print("[INFO] CVAT coordinates:")
print(
    "xtl =", box.get("xtl"),
    "ytl =", box.get("ytl"),
    "xbr =", box.get("xbr"),
    "ybr =", box.get("ybr")
)

[INFO] Image: images/01.png
[INFO] Size: 199 x 174
[INFO] Label: pig_face
[INFO] CVAT coordinates:
xtl = 59.50 ytl = 18.27 xbr = 142.30 ybr = 109.00


In [5]:
image_width = int(first_image.get("width"))
image_height = int(first_image.get("height"))

xtl = float(box.get("xtl"))
ytl = float(box.get("ytl"))
xbr = float(box.get("xbr"))
ybr = float(box.get("ybr"))

x_center = ((xtl + xbr) / 2) / image_width
y_center = ((ytl + ybr) / 2) / image_height
width = (xbr - xtl) / image_width
height = (ybr - ytl) / image_height

print("[INFO] YOLO coordinates:")
print("x_center:", x_center)
print("y_center:", y_center)
print("width:", width)
print("height:", height)

[INFO] YOLO coordinates:
x_center: 0.5070351758793971
y_center: 0.36571839080459767
width: 0.4160804020100503
height: 0.5214367816091954


In [6]:
from pathlib import Path

labels_dir = Path("yolo_labels")
labels_dir.mkdir(exist_ok=True)

class_mapping = {
    "pig_face": 0
}

for image in root.findall("image"):

    image_name = Path(image.get("name")).stem
    image_width = int(image.get("width"))
    image_height = int(image.get("height"))

    yolo_lines = []

    for box in image.findall("box"):

        label = box.get("label")
        class_id = class_mapping[label]

        xtl = float(box.get("xtl"))
        ytl = float(box.get("ytl"))
        xbr = float(box.get("xbr"))
        ybr = float(box.get("ybr"))

        x_center = ((xtl + xbr) / 2) / image_width
        y_center = ((ytl + ybr) / 2) / image_height
        width = (xbr - xtl) / image_width
        height = (ybr - ytl) / image_height

        yolo_line = (
            f"{class_id} "
            f"{x_center:.6f} "
            f"{y_center:.6f} "
            f"{width:.6f} "
            f"{height:.6f}"
        )

        yolo_lines.append(yolo_line)

    label_file = labels_dir / f"{image_name}.txt"
    label_file.write_text("\n".join(yolo_lines))

print(f"[SUCCESS] Converted {len(list(root.findall('image')))} images.")
print(f"[INFO] YOLO labels saved to: {labels_dir}")

[SUCCESS] Converted 27 images.
[INFO] YOLO labels saved to: yolo_labels


In [7]:
label_file = labels_dir / "01.txt"

print("[INFO] Contents of 01.txt:")
print(label_file.read_text())

[INFO] Contents of 01.txt:
0 0.507035 0.365718 0.416080 0.521437


In [8]:
label_files = list(labels_dir.glob("*.txt"))

print("[INFO] Number of YOLO label files:", len(label_files))
print("[INFO] First 5 files:")

for file in label_files[:5]:
    print(" -", file.name)

[INFO] Number of YOLO label files: 27
[INFO] First 5 files:
 - 09.txt
 - 06.txt
 - 03.txt
 - 08.txt
 - 11.txt
